# Load and import data

## Imports and setup

In [1]:
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
import numpy as np 
import pandas as pd
from cmcrameri import cm
from matplotlib.ticker import PercentFormatter
import socket 
import os
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt

In [2]:


arial_path = "/wolke_scratch/dnikolo/MSGI/plotting/Publication_figures/arial/ARIAL.TTF"

fm.fontManager.addfont(arial_path)
font_name = fm.FontProperties(fname=arial_path).get_name()

mpl.rcParams.update({
    "font.family": font_name,
    "font.size": 6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [3]:

# %matplotlib inline
# plt.rc('font',family='helvetica')
# plt.rcParams.update({"font.size":6})
# font= fm.FontProperties(family='helvetica')
# mpl.rcParams['pdf.fonttype'] = 42
# mpl.rcParams['ps.fonttype'] = 42
# cmcrameri.cm.show_cmaps()

## Model data

In [4]:
config_cesm_cal = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/CESM_output/test_01/2000_tracking/01_01.yaml")
config_cesm_ootb = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/CESM_output/ootb/2000_tracking/01_01.yaml")
config_icon_rl_wbf = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/ICON_output/no_nudging/R2B4/wbf_v3/2007_tracking/01_01.yaml")
config_icon_i_wbf = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/ICON_output/no_nudging/R2B4/wbf_v1/2007_tracking/01_01.yaml")
config_icon_p3 = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/ICON_output/sedim_search/R2B4/p3_lite/2007_tracking/01_01.yaml")

In [5]:
loader_cesm_cal = ChunkLoader([2000],config_cesm_cal)
loader_cesm_ootb = ChunkLoader([2000],config_cesm_ootb)

Cloud_fps: {2000: '/nfs/ch4/wolke_scratch/dnikolo/CESM_Data/test_01/postprocessing_output/Final_results/2000_all.parquet'}

Missing cloud coverage for year 2000:
  month=12, half=16-end, temp=-36
  month=12, half=16-end, temp=-30
  month=12, half=16-end, temp=-24
  month=12, half=16-end, temp=-18
  month=12, half=16-end, temp=-12
  month=12, half=16-end, temp=-6
Cloud_fps: {2000: '/nfs/ch4/wolke_scratch/dnikolo/CESM_Data/ootb/postprocessing_output/Final_results/2000_all.parquet'}

Missing cloud coverage for year 2000:
  month=12, half=16-end, temp=-36
  month=12, half=16-end, temp=-30
  month=12, half=16-end, temp=-24
  month=12, half=16-end, temp=-18
  month=12, half=16-end, temp=-12
  month=12, half=16-end, temp=-6


In [6]:
loader_icon_rl = ChunkLoader([2007],config_icon_rl_wbf)
loader_icon_i = ChunkLoader([2007],config_icon_i_wbf)
loader_icon_p3 = ChunkLoader([2007],config_icon_p3)

Cloud_fps: {2007: '/nfs/ch4/wolke_scratch/dnikolo/ICON_Data/no_nudging/R2B4/wbf_v3/postprocessing_output/Final_results/2007_all.parquet'}

Missing cloud coverage for year 2007:
  month=04, half=16-end, temp=-36
  month=04, half=16-end, temp=-30
  month=04, half=16-end, temp=-24
  month=04, half=16-end, temp=-18
  month=04, half=16-end, temp=-12
  month=04, half=16-end, temp=-6
Cloud_fps: {2007: '/nfs/ch4/wolke_scratch/dnikolo/ICON_Data/no_nudging/R2B4/wbf_v1/postprocessing_output/Final_results/2007_all.parquet'}

Missing cloud coverage for year 2007:
  month=04, half=16-end, temp=-36
  month=04, half=16-end, temp=-30
  month=04, half=16-end, temp=-24
  month=04, half=16-end, temp=-18
  month=04, half=16-end, temp=-12
  month=04, half=16-end, temp=-6
Cloud_fps: {2007: '/nfs/ch4/wolke_scratch/dnikolo/ICON_Data/sedim_search/R2B4/p3_lite/postprocessing_output/Final_results/2007_all.parquet'}

Missing cloud coverage for year 2007:
  month=01, half=01-15, temp=-36
  month=01, half=01-15, tem

In [7]:
glac_icon_rl = loader_icon_rl.glac_chunk
glac_icon_i = loader_icon_i.glac_chunk
glac_icon_p3 = loader_icon_p3.glac_chunk

In [8]:
glac_cesm_cal = loader_cesm_cal.glac_chunk
glac_cesm_ootb = loader_cesm_ootb.glac_chunk

In [9]:
def return_glac_df(a):
    return a[1]
# loader_dd = ChunkLoader(years,config_dd,load_init_chunk=False)
# glac_dfs_dd = loader_dd.execute_analysis(return_glac_df,load_clouds=False)
# df_glac_dd = pd.concat(glac_dfs_dd,ignore_index=True)

## Dardar data

In [10]:
bulk_data_colocated= pd.read_parquet("/wolke_scratch/dnikolo/MSGI/plotting/Publication_figures/data/observations/bulk_data_colocated.parquet")
bulk_data_filtered = pd.read_parquet("/wolke_scratch/dnikolo/MSGI/plotting/Publication_figures/data/observations/bulk_data_filtered.parquet")
bulk_data_colocated["ct_cph"] = bulk_data_colocated.apply(lambda row: row["cph_bulk_means"][0],axis=1)-1
bulk_data_filtered["ct_cph"] = bulk_data_filtered.apply(lambda row: row["cph_bulk_means"][0],axis=1)-1

In [11]:
find_ct_claas_if = lambda row: row["val_pix_claas_if_hist"][int((row["colocation_time"]-row["track_start_time"])/pd.Timedelta(minutes=15))]
find_ct_dardar_if = lambda row: row["cph_bulk_means"][0]-1
bulk_data_colocated["claas_if"] = bulk_data_colocated.apply(find_ct_claas_if,axis=1)
bulk_data_filtered["claas_if"] = bulk_data_filtered.apply(find_ct_claas_if,axis=1)
bulk_data_colocated["ct_cph"] = bulk_data_colocated.apply(find_ct_dardar_if,axis=1)
bulk_data_filtered["ct_cph"] = bulk_data_filtered.apply(find_ct_dardar_if,axis=1)

In [12]:
eps=1e-5
def get_dd_pre_peak_if(row: pd.Series,idx: int):
    if_hist = row["val_ice_frac_hist"]
    if ((row["Glac_peak_ind"]-idx)>=0) and ((row["Glac_peak_ind"]-idx)<len(if_hist)):
        if if_hist[row["Glac_peak_ind"]-idx]>=-eps:
            if int((row["colocation_time"]-row["track_start_time"])/pd.Timedelta(minutes=15))==(row["Glac_peak_ind"]-idx):
                return row["val_ice_frac_hist"][row["Glac_peak_ind"]-idx]
    return np.nan

for pre_peak_idx in range(-5, 5):
    bulk_data_colocated[f"val_{pre_peak_idx}_prepeak_if"] = bulk_data_colocated.apply(
            get_dd_pre_peak_if,
            axis=1,
            args=(pre_peak_idx,)
        )
    bulk_data_filtered[f"val_{pre_peak_idx}_prepeak_if"] = bulk_data_filtered.apply(
            get_dd_pre_peak_if,
            axis=1,
            args=(pre_peak_idx,)
        )

In [13]:
df_glac_dd = bulk_data_colocated[bulk_data_colocated["avg_ctp"]>440]
df_glac_dd = df_glac_dd[(df_glac_dd["val_ctt"]>273.15-38) & (df_glac_dd["val_ctt"]<273.15)]
df_glac_dd = df_glac_dd[df_glac_dd["ct_depth"]<1000]
df_glac_dd = df_glac_dd[df_glac_dd["avg_ctt"]>273.15-36]
df_glac_dd = df_glac_dd[df_glac_dd["avg_ctt"]<273.15]

In [14]:
def print_prepeak_stats(df):    
    for i in range(2,-3,-1):
        group = df[
            (df.rel_ind >= -3) & 
            (df.rel_ind <= 3) & 
            df[f"val_{i}_prepeak_if"].notna()
        ].groupby("rel_ind")
        print(-i, group[f"val_{i}_prepeak_if"].mean().values, group[f"val_{i}_prepeak_if"].median().values, group[f"val_{i}_prepeak_if"].count().values)


In [15]:
df_glac_dd.track_start_time.dt.year.value_counts()

track_start_time
2007    708
2009    596
2010    581
2008    488
2006    144
Name: count, dtype: int64

In [16]:
print_prepeak_stats(df_glac_dd)

-2 [0.82370424] [0.88110697] [67]
-1 [0.83654559] [0.90476191] [67]
0 [0.86423796] [0.9117228] [52]
1 [0.82130527] [0.87967685] [50]
2 [0.84866054] [0.91369047] [43]
